# Import

In [ ]:
from __future__ import annotations
import json
import matplotlib.pyplot as plt
import argparse
import time
from pathlib import Path

import numpy as np
from PIL.ImageOps import grayscale

#from PythonProject.rust_bridge import RBFModelRust
from dataset_loader import DEFAULT_DATASET_ROOT, load_labeled_image_dataset, stratified_split, _load_image_vector, DEFAULT_CLASS_NAMES
from rust_bridge import MLPRust, OVRLinearClassifier, TaskMode, OVRRBF


# Fonction

In [ ]:
#---Parser pour utiliser le code avec des commandes---
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Train the Rust linear baseline and MLP on the game screenshot dataset."
    )
    parser.add_argument(
        "--root",
        type=Path,
        default=DEFAULT_DATASET_ROOT,
        help="Root directory containing FPS, METROIDVANIA and MOBA subfolders.",
    )
    parser.add_argument("--width", type=int, default=8, help="Resized image width.")
    parser.add_argument("--height", type=int, default=6, help="Resized image height.")
    parser.add_argument("--grayscale", dest="grayscale", action="store_true", help="Use grayscale images.")
    parser.add_argument("--rgb", dest="grayscale", action="store_false", help="Use RGB images.")
    parser.add_argument("--test-ratio", type=float, default=0.2, help="Fraction used for test split.")
    parser.add_argument("--seed", type=int, default=42, help="Random seed for the split.")
    parser.add_argument("--linear-lr", type=float, default=0.01, help="Learning rate for the linear baseline.")
    parser.add_argument(
        "--linear-steps",
        "--linear-epochs",
        dest="linear_steps",
        type=int,
        default=20_000,
        help="Random training steps for the linear baseline.",
    )
    parser.add_argument("--mlp-lr", type=float, default=0.01, help="Learning rate for the MLP.")
    parser.add_argument(
        "--mlp-steps",
        "--mlp-epochs",
        dest="mlp_steps",
        type=int,
        default=50_000,
        help="Random training steps for the naive MLP.",
    )
    parser.add_argument(
        "--mlp-layers",
        type=int,
        nargs="*",
        default=[16, 8],
        help="Hidden layer sizes for the MLP. Example: --mlp-layers 16 8",
    )
    parser.set_defaults(grayscale=True)
    return parser.parse_args()

#---Affichage des résultats---
def _print_results(
    title: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: tuple[str, ...],
) -> None:
    global accuracy
    accuracy= float(np.mean(np.all(y_true == y_pred, axis=1)))
    confusion = _confusion_matrix(y_true, y_pred, len(class_names))

    # display of one result block
    print(f"{title}:")
    print(f"Accuracy: {accuracy:.3f}")
    print("Matrice de confusion (lignes=reel, colonnes=predit) :")
    print("       " + " ".join(f"{name[:5]:>6}" for name in class_names))
    for row_index, row in enumerate(confusion):
        print(f"{class_names[row_index][:5]:>5} " + " ".join(f"{value:>6}" for value in row))

#---Matrice de confusion---
def _confusion_matrix(y_true: np.ndarray,
                      y_pred: np.ndarray,
                      class_count: int
                      ) -> np.ndarray:
    matrix = np.zeros((class_count, class_count), dtype=np.int32)
    true_indices = np.argmax(y_true, axis=1)
    pred_indices = np.argmax(y_pred, axis=1)

    for true_index, pred_index in zip(true_indices, pred_indices):
        matrix[true_index, pred_index] += 1

    return matrix


def calcul_loss(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    difference = y_true.reshape(-1) - y_pred.reshape(-1)
    return float(np.mean(difference * difference))


def loss(model, x_train, y_train, x_test, y_test) -> dict:
    loss_avant_train = calcul_loss(y_train, np.zeros_like(y_train))
    loss_avant_test = calcul_loss(y_test, np.zeros_like(y_test))

    pred_train = model.prediction(x_train)
    pred_test = model.prediction(x_test)

    loss_apres_train = calcul_loss(y_train, pred_train)
    loss_apres_test = calcul_loss(y_test, pred_test)

    return {
        "before": {"train_loss": loss_avant_train, "test_loss": loss_avant_test},
        "after": {"train_loss": loss_apres_train, "test_loss": loss_apres_test},
    }


def affichage_loss_rbf(result: dict, title: str = "RBF - loss avant/après entraînement") -> None:
    labels = ["train", "test"]
    before = [result["before"]["train_loss"], result["before"]["test_loss"]]
    after = [result["after"]["train_loss"], result["after"]["test_loss"]]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots()
    ax.bar(x - width/2, before, width, label="avant")
    ax.bar(x + width/2, after, width, label="après")

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("loss")
    ax.set_title(title)
    ax.legend()
    plt.show()
    plt.clf()


def train_avec_loss(model : OVRLinearClassifier, x_train, y_train, x_test, y_test, total_epochs: int, loss_tout_les: int = 1) -> dict:
    log = {"epoch": [], "train_loss": [], "test_loss": []}

    for epoch in range(0,total_epochs, loss_tout_les):
        model.fit(x_train, y_train, epochs=loss_tout_les)

        pred_train = model.prediction(x_train)
        pred_test = model.prediction(x_test)
        log["epoch"].append(epoch)
        log["train_loss"].append(calcul_loss(y_train, pred_train))
        log["test_loss"].append(calcul_loss(y_test, pred_test))

    return log

def affichage_loss_lineaire(log: dict, title: str):
    plt.plot(log["epoch"], log["train_loss"], label="train")
    plt.plot(log["epoch"], log["test_loss"], label="test")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.show()
    plt.clf()

# Code

In [ ]:
etat_grayscale = True #True ou false, activation de la nuance de gris
chemin = r"G:\Mon Drive\Dataset projet annuel"
width = int(40 * 16/9)
height = 40

#---Création du Dataset transformer---
bundle = load_labeled_image_dataset(root=chemin, image_size=(width, height), grayscale=etat_grayscale)

#--Séparation des données en test et train---
split_bundle = stratified_split(bundle, test_ratio=0.2, seed=42)

#---Affichage des informations---
print("=== Dataset ===")
print(f"Chemin: {chemin}")
print(f"Classes: {', '.join(bundle.class_names)}")
print(f"Nb images par classe: {bundle.counts_by_class}")
color_mode = "grayscale" if etat_grayscale else "rgb"
print(f"Format apres pretraitement: {width}x{height} {color_mode}")
print(f"Nb features: {bundle.x.shape[1]}")
print(f"Nb images train: {split_bundle.x_train.shape[0]}")
print(f"Nb images test: {split_bundle.x_test.shape[0]}")
print(f"Images ignorees: {len(bundle.skipped_paths)}")
if bundle.skipped_paths:
    for skipped_path in bundle.skipped_paths[:5]:
        print(f"  - {skipped_path.name}")
print()

# Entrainement des modèles


# Linéaire

In [ ]:
#---Entrainement modèle linéaire---
start_time_all = time.perf_counter()
epochs = 1_000 * split_bundle.x_train.shape[0]             #Nombre de fois où le modèle s'entrainera sur l'ensemble des données
pas_apprentissage = 0.0001           #Pas d'apprentissage du modèle linéaire
batch_size_like = 100000             #Calcul du loss tout les x epoch
liste_seed = [12, 24, 42, 5, 59]

tous_les_train_loss = []
tous_les_test_loss = []
liste_accuracy = []



print(f"Allé retour Python <-> Rust :{epochs / batch_size_like}")
for i in liste_seed:
    start_time = time.perf_counter()
    linear = OVRLinearClassifier(
            input_dim=split_bundle.x_train.shape[1],
            output_dim=len(bundle.class_names),
            learning_rate=pas_apprentissage,
            seed= i
        )
    log_loss = train_avec_loss(linear, split_bundle.x_train, split_bundle.y_train, split_bundle.x_test, split_bundle.y_test, total_epochs=epochs, loss_tout_les=batch_size_like)
    linear_predictions = linear.predict_labels(split_bundle.x_test)

    #---Résultats---
    print(f"=== Resultats seed {i} ===")
    _print_results("Linear", split_bundle.y_test, linear_predictions, bundle.class_names)
    print()
    print(f"Temps: {time.perf_counter() - start_time:.2f}s")
    linear.set_accuracy(accuracy=accuracy)
    affichage_loss_lineaire(log_loss, f"Loss modèle linéaire {i}")

    linear.sauvegarde("save_model/model_linear1.json", log=log_loss, epochs=epochs)
    linear.close()




print(f"Temps total: {time.perf_counter() - start_time_all:.2f}s")


In [ ]:
#linear.sauvegarde("save_model/model_linear1.json", log=log_loss, accuracy= accuracy, epochs=epochs)

#linear.close()

In [ ]:
new_linear = OVRLinearClassifier.charge("save_model/model_linear1.json")


image = _load_image_vector(r"C:\Users\theot\Downloads\fps.jpg", (int(40 * 16/9), 40), True)


#---Prédiction---
scores = new_linear.predict_labels(image)
predicted_index = int(np.argmax(scores[0]))
predicted_class = DEFAULT_CLASS_NAMES[predicted_index]

print(f"Scores bruts : {scores[0]}")
print(f"Classe prédite : {predicted_class}")

new_linear.close()

# RBF

In [ ]:
#---Entrainement modèle RBF---
start_time_all = time.perf_counter()
mouvement_max = 0.0001      #Mouvement accepté lors de lloyd
nb_cluster = 200            #Nombre de cluster pour lloyd
max_loop = 1000             #Nombre maximul de tour
gamma = 0.01
liste_seed = [12, 24, 42, 5, 59]

for i in liste_seed:
    start_time = time.perf_counter()
    RBF = OVRRBF(
            input_dim=split_bundle.x_train.shape[1],
            nb_cluster=nb_cluster,
            output_dim=len(bundle.class_names),
            seed=i
        )

    RBF.entrainement(split_bundle.x_train,split_bundle.y_train,mouvement_max,max_loop, gamma)

    pred = RBF.prediction(split_bundle.x_test)


    #---Résultats---
    print(f"=== Resultats seed {i} ===")
    _print_results("RBF", split_bundle.y_test, pred, bundle.class_names)
    print()
    print()
    print(f"Temps total: {time.perf_counter() - start_time:.2f}s")


    loss_rbf = loss(RBF, split_bundle.x_train, split_bundle.y_train, split_bundle.x_test, split_bundle.y_test)
    RBF.set_accuracy(accuracy=accuracy)
    affichage_loss_rbf(loss_rbf, title=f"RBF loss {i}")
    RBF.sauvegarde(
    "save_model/model_rbf2.json", log_loss= log_loss,
     nb_cluster=nb_cluster,mouvement_max=mouvement_max,max_loop=max_loop,gamma=gamma,
    )


    RBF.close()


print(f"Temps total: {time.perf_counter() - start_time_all:.2f}s")


In [ ]:
#RBF.sauvegarde(
#    "save_model/model_rbf2.json", log_loss= log_loss,
#    nb_cluster=nb_cluster,mouvement_max=mouvement_max,max_loop=max_loop,gamma=gamma, accuracy=accuracy
#)


#RBF.close()

In [ ]:
new_RBF = OVRRBF.charge("save_model/model_rbf2.json")

#with open("save_model/model_rbf2.json") as f:
#    hp = json.load(f)["parametres"]



image = _load_image_vector(r"C:\Users\theot\Downloads\images.jpg", (int(40 * 16/9), 40), True)



#---Prédiction---
scores = new_RBF.prediction(image)
predicted_index = int(np.argmax(scores[0]))
predicted_class = DEFAULT_CLASS_NAMES[predicted_index]

print(f"Scores bruts : {scores[0]}")
print(f"Classe prédite : {predicted_class}")

new_RBF.close()